In [82]:
--Adolfo David Romero
--991555778
--Assignment 3B (bonus)
USE section19
GO

Commands completed successfully.

Total execution time: 00:00:00.003

In [83]:
--Verify and test tables for testing (copy-pasted from previous assingnment)
--Drop tables to start fresh and create tables
DROP TABLE IF EXISTS OrderProd;
DROP TABLE IF EXISTS SalesOrder;
DROP TABLE IF EXISTS Customer;
DROP TABLE IF EXISTS Part;
DROP TABLE IF EXISTS SalesRep;
GO
--Part A
--CREATE tables with rows mentioned in assignment
CREATE TABLE Customer(
    custno INT PRIMARY KEY, 
    custname VARCHAR(50),
    balance DECIMAL(10,2)
);
CREATE TABLE SalesOrder(
    orderno INT PRIMARY KEY, 
    custno INT,
    orderdate DATE, 
    FOREIGN KEY (custno) REFERENCES CUSTOMER(custno)
);
CREATE TABLE OrderProd(
    orderno INT, 
    partno INT,
    orderqty INT, 
    orderprice DECIMAL(10,2),
    PRIMARY KEY (orderno, partno),
    FOREIGN KEY (orderno) REFERENCES SalesOrder(orderno)
);

--Part B
CREATE TABLE Part (
    partno INT PRIMARY KEY,
    partname VARCHAR(50),
    unitprice DECIMAL(10,2),
    onhand INT
);
CREATE TABLE SalesRep (
    srepno INT PRIMARY KEY,
    srepname VARCHAR(50),
    srepstreet VARCHAR(50),
    srepcity VARCHAR(30),
    srepprov VARCHAR(30),
    sreppcode VARCHAR(30),
    totcomm DECIMAL(10,2),
    commrate DECIMAL(4,2)
);

--LOAD tables with fake data
INSERT INTO Customer (custno, custname, balance) VALUES 
(1, 'David Romero', 20.00), 
(2, 'Robert Marley', 400.00), 
(3, 'Brent Perteron', 00.00), 
(4, 'Magdin Stoica', 8045.57), 
(5, 'Bob Bobbert', 94.84),
(6, 'Testy McGee', 0.00); --Test case

INSERT INTO SalesOrder (orderno, custno, orderdate) VALUES 
(1,1,'2025-01-01'),
(2,2,'2025-01-01'),
(3,2,'2025-03-11'),
(4,3,'1999-02-19'),
(5,5,'2025-01-03');

INSERT INTO OrderProd (orderno, partno, orderqty, orderprice) VALUES
(1, 1, 2, 10.00),  
(2, 2, 1, 200.00), 
(3, 3, 5, 30.00),  
(4, 4, 1, 10.00),  
(5, 5, 3, 15.00);  

--part B
INSERT INTO Part(partno,partname,unitprice,onhand) VALUES
(1, '12-foot hammer', 30.00, 3),
(2, 'Tape', 5.24, 300),
(3, 'Google glasses', 299.99, 1),
(4, '4 tons of gravel', 500.00, 4),
(5, 'Brick', 59.99, 50);

INSERT INTO SalesRep(srepno,srepname,srepstreet,srepcity,srepprov,sreppcode,totcomm,commrate) VALUES
(1, 'George','Appleseed Rd','Toronto','ON','L6M 0G4',200.00,12.2),
(2, 'Samantha','Penny Lane','Liverpool','ON','L3M 0E3',50.00,10.2),
(3, 'John','Abbey Rd','London','ON','E6M 0T1',100.00,5.00),
(4, 'Paul','Winston st','Oakville','ON','Y2L 0U4',0.00, 40.00),
(5, 'Ringo','street st','Victoria','BC','Q4U 7R1',13.12, 12.12);

--Confirm tables
SELECT * FROM Customer
SELECT * FROM SalesOrder
SELECT * FROM OrderProd
SELECT * FROM Part
SELECT * FROM SalesRep


Commands completed successfully.

(6 rows affected)

(5 rows affected)

(5 rows affected)

(5 rows affected)

...

custno,custname,balance
1,David Romero,20.00
2,Robert Marley,400.00
3,Brent Perteron,0.00
4,Magdin Stoica,8045.57
5,Bob Bobbert,94.84
6,Testy McGee,0.00


orderno,custno,orderdate
1,1,2025-01-01
2,2,2025-01-01
3,2,2025-03-11
4,3,1999-02-19
5,5,2025-01-03


orderno,partno,orderqty,orderprice
1,1,2,10.00
2,2,1,200.00
3,3,5,30.00
4,4,1,10.00
5,5,3,15.00


partno,partname,unitprice,onhand
1,12-foot hammer,30.00,3
2,Tape,5.24,300
3,Google glasses,299.99,1
4,4 tons of gravel,500.00,4
5,Brick,59.99,50


srepno,srepname,srepstreet,srepcity,srepprov,sreppcode,totcomm,commrate
1,George,Appleseed Rd,Toronto,ON,L6M 0G4,200.00,12.20
2,Samantha,Penny Lane,Liverpool,ON,L3M 0E3,50.00,10.20
3,John,Abbey Rd,London,ON,E6M 0T1,100.00,5.00
4,Paul,Winston st,Oakville,ON,Y2L 0U4,0.00,40.00
5,Ringo,street st,Victoria,BC,Q4U 7R1,13.12,12.12


# **PART A - CURSORS**

In [84]:
-- 1
DROP PROCEDURE AdjustAccountBalances
GO

CREATE PROCEDURE AdjustAccountBalances
AS 
BEGIN
    DECLARE @custno INT, @totalAmount DECIMAL(10,2); -- Store customer number and total transaction amount 
    DECLARE customer_cursor CURSOR FOR SELECT custno FROM Customer --customer cursor 

    OPEN customer_cursor; --executes above 'SELECT custno FROM Customer'
    FETCH NEXT FROM customer_cursor INTO @custno; --fetch the first customer

    WHILE @@FETCH_STATUS = 0 --loop through all customers (Iterate through each customer's sales transactions.)
    BEGIN

        --calculates total sales for the customer, cost per product
        SELECT @totalAmount = ISNULL(SUM(op.orderqty * op.orderprice), 0) -- ISNULL is used in case cust has NO orders (potential edge case). Resurns 0 if null
        FROM SalesOrder so --use alliases to acccess
        JOIN ORDERPROD op ON so.orderno = op.orderno
        WHERE so.custno = @custno

        PRINT CONCAT('UPDATED TOTAL AMOUNT: ',@totalAmount);

        --update customer balance col
        UPDATE CUSTOMER
        SET balance = balance - @totalAmount --subtract balance using var
        WHERE custno = @custno 

        PRINT CONCAT('UPDATED CUSTOMER: ',@custno);

        FETCH NEXT FROM customer_cursor INTO @custno; -- Move to next customer
    END;

    --clean up crew
    CLOSE customer_cursor;
    DEALLOCATE customer_cursor;

END;

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.015

In [85]:
--Test Cursor

--Customer 6 has an initial balance of 0 for a test case
SELECT * FROM Customer --before
SELECT * FROM SalesOrder
SELECT * FROM OrderProd

EXEC AdjustAccountBalances;

SELECT * FROM Customer --after


(6 rows affected)

(5 rows affected)

(5 rows affected)

UPDATED TOTAL AMOUNT: 20.00

(1 row affected)

...

Total execution time: 00:00:00.033

custno,custname,balance
1,David Romero,20.00
2,Robert Marley,400.00
3,Brent Perteron,0.00
4,Magdin Stoica,8045.57
5,Bob Bobbert,94.84
6,Testy McGee,0.00


orderno,custno,orderdate
1,1,2025-01-01
2,2,2025-01-01
3,2,2025-03-11
4,3,1999-02-19
5,5,2025-01-03


orderno,partno,orderqty,orderprice
1,1,2,10.00
2,2,1,200.00
3,3,5,30.00
4,4,1,10.00
5,5,3,15.00


custno,custname,balance
1,David Romero,0.00
2,Robert Marley,50.00
3,Brent Perteron,-10.00
4,Magdin Stoica,8045.57
5,Bob Bobbert,49.84
6,Testy McGee,0.00


# **PART B - DYNAMIC SQL**

Commands completed successfully.

Total execution time: 00:00:00